In [10]:
%load_ext autoreload
%autoreload 2


import optuna
import pandas as pd


from TrailPredictor.dataset import Dataset
from TrailPredictor.modeling.heart_rate import (
    REFERENCE_CUTS,
    HeartRateConfig,
    cuts_to_zones,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
dataset = Dataset.load(min_distance=40, max_distance=100, segment_width=500)
dataset.target = "pace"  # set before building the model: the result is saved per target

config = HeartRateConfig(
    n_trials=50,
    max_trials_factor=20,
    zone_counts=(2, 3, 4, 5, 6),
    seed=42,
    n_splits=5,
)
# reloads the saved trials of this exact configuration, if any
model = dataset.modeling.heart_rate(config)

# plotter = dataset.plot

## Heart-rate zone cuts optimisation (Optuna)

Only the zone definition changes from one trial to the next: for N zones, Optuna suggests N-1
increasing cuts, the zones are rebuilt on a copy of the dataset (`Dataset.with_heart_rate_zones`),
then the pipeline is evaluated with `GroupKFold(config.n_splits)` grouped by `activity_id`.
One study per N in `config.zone_counts`, then N is chosen with the 1-SE rule.

The trials are saved under `data/computed/models/heart_rate/<target>/<config hash>.csv`
(with the configuration as JSON next to it) and reloaded when the same `HeartRateConfig` is used again.

In [12]:
# Run the studies only when no result is saved for this configuration.
# The Optuna studies are not saved: `studies` is empty after a reload.
studies = model.optimize_zones() if model.result is None else {}

In [13]:
# Reference: current HEART_RATE_ZONES cuts, as contiguous zones from 0 to 250 bpm
REFERENCE_RMSE, REFERENCE_SE = model.evaluate_cuts_zones(REFERENCE_CUTS)
model.reference_rmse = REFERENCE_RMSE  # reference line of model.plot

f"RMSE de référence (HEART_RATE_ZONES) : {REFERENCE_RMSE:.4f} ± {REFERENCE_SE:.4f}"

'RMSE de référence (HEART_RATE_ZONES) : 1.9194 ± 0.2747'

In [14]:
# Summary built from the saved trials (model.result), so it also works after a reload
trial_counts = pd.crosstab(model.result["n_zones"], model.result["state"]).reindex(
    columns=["COMPLETE", "PRUNED"], fill_value=0
)
summary = model.plot.best_trials[["cuts", "rmse", "rmse_se"]].assign(
    n_complete=trial_counts["COMPLETE"],
    n_pruned=trial_counts["PRUNED"],
    reference_rmse=REFERENCE_RMSE,
    reference_se=REFERENCE_SE,
    delta_vs_reference=lambda df: df["rmse"] - REFERENCE_RMSE,
)

# Règle du 1-SE : le plus petit N dont le RMSE reste sous (meilleur RMSE + son erreur standard)
best = summary.nsmallest(1, "rmse").iloc[0]  # row with the lowest RMSE
best_n = best.name
one_se_threshold = best["rmse"] + best["rmse_se"]
SELECTED_N = summary.index[summary["rmse"] <= one_se_threshold].min()
SELECTED_ZONES = cuts_to_zones(summary["cuts"].loc[SELECTED_N])

display(
    summary.assign(
        **{
            "RMSE ± SE": summary["rmse"].map("{:.4f}".format)
            + " ± "
            + summary["rmse_se"].map("{:.4f}".format),
            "référence ± SE": f"{REFERENCE_RMSE:.4f} ± {REFERENCE_SE:.4f}",
            "retenu (1-SE)": summary.index == SELECTED_N,
        }
    )[
        [
            "cuts",
            "RMSE ± SE",
            "référence ± SE",
            "delta_vs_reference",
            "n_complete",
            "n_pruned",
            "retenu (1-SE)",
        ]
    ]
)
f"Meilleur N = {best_n}, seuil 1-SE = {one_se_threshold:.4f} -> N retenu = {SELECTED_N} : {SELECTED_ZONES}"

,cuts,RMSE ± SE,référence ± SE,delta_vs_reference,n_complete,n_pruned,retenu (1-SE)
n_zones,,,,,,,
2,[164],1.9179 ± 0.1927,1.9194 ± 0.2747,-0.001501,50,0,True
3,"[151, 164]",1.8363 ± 0.2335,1.9194 ± 0.2747,-0.083093,50,1,False
4,"[151, 163, 175]",1.8407 ± 0.2343,1.9194 ± 0.2747,-0.078664,50,5,False
5,"[151, 165, 178, 188]",1.8427 ± 0.2333,1.9194 ± 0.2747,-0.076734,50,16,False
6,"[120, 131, 151, 159, 164]",1.8410 ± 0.2500,1.9194 ± 0.2747,-0.078411,50,10,False


"Meilleur N = 3, seuil 1-SE = 2.0698 -> N retenu = 2 : {'Z0': (0, 164), 'Z1': (164, 250)}"

In [15]:
for n in model.result["n_zones"].unique():
    model.plot.trials_zones(n).show()

In [16]:
model.plot.best_zones_per_n().show()

In [17]:
# RMSE en fonction de chaque paramètre (c1, gap_k) pour le N retenu
# needs the Optuna study, only in memory right after optimize_zones()
if SELECTED_N in studies:
    optuna.visualization.plot_slice(studies[SELECTED_N]).show()
else:
    display("No study in memory (trials reloaded from disk): no slice plot.")

'No study in memory (trials reloaded from disk): no slice plot.'